# Módulo modelado: conjunto, variables y modelos
Lo usan los notebooks 19 a 22 con `%run -i modulos/modelado.ipynb` (después de `modulos/comun.ipynb`).

- **Conjunto:** los procesos no penales **con movimiento** en 2023, es decir, con al menos una causa atendida (`dataset_analitico_curado`,
  una fila por tipo de proceso × territorio × materia). El filtro usa solo lo conocido al empezar el año: no se descartan
  filas mirando las resueltas, porque eso sería elegir los ejemplos con la respuesta.
- **Objetivo:** `flag_acumula_mora`, verdadero cuando en 2023 se **resolvieron menos causas de las que ingresaron** (tasa de resolución < 1,
  notebook 10): por la identidad contable, ese año el stock crece.
- **Variables:** solo lo que se conoce **al empezar el año** (stock heredado, causas que entran, materia, territorio, juzgados y personal).
  Todo lo que depende de las causas **resueltas** queda fuera, porque es la respuesta disfrazada (*fuga de información*).
- **Ingeniería:** 8 variables calculadas fila por fila (sin promedios de grupo, para que el despliegue pueda recalcularlas con una fórmula).

In [ ]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (AdaBoostClassifier, BaggingClassifier, GradientBoostingClassifier,
                              RandomForestClassifier, StackingClassifier)
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

SEMILLA = 42
MODELO_DIR = CURATED / "modelo"
MODELO_DIR.mkdir(parents=True, exist_ok=True)
IMG = RAIZ / "web" / "img"
OBJETIVO = "flag_acumula_mora"
CV5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

# Columnas que identifican la fila; no entran al modelo.
IDENTIFICADORES = ["cuadro_origen", "pagina_pdf", "tipo_proceso", "territorio"]

CATEGORICAS = ["materia_homologada", "ambito", "departamento_derivado"]
NUMERICAS_ORIGINALES = [
    "pendientes_inicio_efectivo",       # stock heredado de 2022 (en civil, las readecuadas a la Ley 439)
    "nuevas_ingresadas",
    "recibidas_excusa_recusacion",
    "preliminares_formalizados",        # solo civil; en otras materias esa forma de ingreso no existe -> 0
    "cautelares_formalizados",          # idem
    "ingresos_totales",                 # todas las formas de ingreso del año
    "atendidas",                        # stock heredado + ingresos
    "num_juzgados_pagina",              # juzgados de esa materia en ese territorio
    "recurso_juzgados_publicados",
    "recurso_tribunales_publicados",
    "recurso_conciliadores_publicados",
    "personal_items_mujer",
    "personal_items_varon",
    "personal_items_acefalias",         # puestos vacantes
    "personal_items_total",
    "personal_remun_total",
]
CEROS_ESTRUCTURALES = ["preliminares_formalizados", "cautelares_formalizados"]
ORIGINALES = CATEGORICAS + NUMERICAS_ORIGINALES

# Variables nuevas -> columnas originales que necesitan (el despliegue las recalcula con la misma fórmula).
DEPENDENCIAS = {
    "log_stock_inicial": ["pendientes_inicio_efectivo"],
    "log_ingresos": ["ingresos_totales"],
    "presion_stock": ["pendientes_inicio_efectivo", "ingresos_totales"],
    "ingresos_por_juzgado": ["ingresos_totales", "num_juzgados_pagina"],
    "stock_por_juzgado": ["pendientes_inicio_efectivo", "num_juzgados_pagina"],
    "prop_acefalias": ["personal_items_acefalias", "personal_items_total"],
    "prop_excusa": ["recibidas_excusa_recusacion", "ingresos_totales"],
    "sin_stock_inicial": ["pendientes_inicio_efectivo"],
}
FORMULAS = {
    "log_stock_inicial": "log(1 + stock heredado)",
    "log_ingresos": "log(1 + ingresos del año)",
    "presion_stock": "stock heredado / (ingresos del año + 1)",
    "ingresos_por_juzgado": "ingresos del año / juzgados",
    "stock_por_juzgado": "stock heredado / juzgados",
    "prop_acefalias": "puestos vacantes / puestos totales",
    "prop_excusa": "recibidas por excusa / (ingresos del año + 1)",
    "sin_stock_inicial": "1 si no había stock heredado",
}
INGENIERIA = list(DEPENDENCIAS)
COMPLETA = ORIGINALES + INGENIERIA

# Columnas excluidas por fuga de información: se calculan con las resueltas o con el stock final.
FUGA = {
    "resueltas": "es parte de la respuesta: el stock crece cuando se resuelve menos de lo que entra",
    "pendientes_fin": "es el resultado del año (stock heredado + ingresos - resueltas)",
    "tasa_resolucion": "resueltas / ingresos",
    "tasa_resolucion_con_readecuadas": "usa resueltas",
    "tasa_resolucion_solo_nuevas": "usa resueltas",
    "tasa_congestion": "atendidas / resueltas",
    "tasa_pendencia": "usa pendientes al final",
    "duracion_estimada_dias": "usa resueltas y pendientes al final",
    "acumulacion_neta_anual": "es el objetivo en número (ingresos - resueltas)",
    "flag_sin_resolucion_anual": "usa resueltas",
    "flag_sin_movimiento": "usa resueltas",
    "estado_gestion": "clasifica la fila según sus resueltas",
    "tasa_congestion_winsorizada": "usa resueltas",
    "duracion_estimada_winsorizada": "usa resueltas",
    "log_resueltas": "usa resueltas",
    "log_pendientes_fin": "usa pendientes al final",
    "es_outlier_congestion_iqr": "marca construida con la tasa de congestión",
    "es_outlier_duracion_iqr": "marca construida con la duración",
}
NOMBRES = {
    "materia_homologada": "materia",
    "ambito": "capital o provincia",
    "departamento_derivado": "departamento",
    "pendientes_inicio_efectivo": "stock heredado",
    "nuevas_ingresadas": "causas nuevas",
    "recibidas_excusa_recusacion": "recibidas por excusa",
    "preliminares_formalizados": "preliminares formalizados",
    "cautelares_formalizados": "cautelares formalizados",
    "ingresos_totales": "ingresos del año",
    "atendidas": "causas atendidas",
    "num_juzgados_pagina": "juzgados (materia y territorio)",
    "recurso_juzgados_publicados": "juzgados del territorio",
    "recurso_tribunales_publicados": "tribunales del territorio",
    "recurso_conciliadores_publicados": "conciliadores del territorio",
    "personal_items_mujer": "puestos ocupados por mujeres",
    "personal_items_varon": "puestos ocupados por varones",
    "personal_items_acefalias": "puestos vacantes",
    "personal_items_total": "puestos totales",
    "personal_remun_total": "remuneración total",
    "log_stock_inicial": "log del stock heredado",
    "log_ingresos": "log de los ingresos",
    "presion_stock": "presión del stock",
    "ingresos_por_juzgado": "ingresos por juzgado",
    "stock_por_juzgado": "stock por juzgado",
    "prop_acefalias": "proporción de vacantes",
    "prop_excusa": "proporción por excusa",
    "sin_stock_inicial": "sin stock heredado",
}

In [ ]:
def construir_variables(df):
    """Las 8 variables nuevas, fila por fila."""
    s = df["pendientes_inicio_efectivo"]
    i = df["ingresos_totales"]
    j = df["num_juzgados_pagina"]
    out = pd.DataFrame(index=df.index)
    out["log_stock_inicial"] = np.log1p(s)
    out["log_ingresos"] = np.log1p(i)
    out["presion_stock"] = s / (i + 1)
    out["ingresos_por_juzgado"] = i / j
    out["stock_por_juzgado"] = s / j
    out["prop_acefalias"] = df["personal_items_acefalias"] / df["personal_items_total"]
    out["prop_excusa"] = df["recibidas_excusa_recusacion"] / (i + 1)
    out["sin_stock_inicial"] = (s == 0).astype(float)
    return out


def cargar_conjunto():
    """Filas del modelo, con los tipos convertidos, los ceros estructurales y las variables nuevas."""
    d = pd.read_parquet(CURATED / "features" / "dataset_analitico_curado.parquet")
    d = d[(d["tipo_elemento_analitico"] == "proceso") & (d["atendidas"] > 0)].copy()
    datos = d[IDENTIFICADORES + ORIGINALES].copy()
    for c in NUMERICAS_ORIGINALES:
        datos[c] = datos[c].astype("float64")        # Int64 con <NA> -> float64 con NaN
    for c in CATEGORICAS:
        datos[c] = datos[c].astype("object")
    datos[CEROS_ESTRUCTURALES] = datos[CEROS_ESTRUCTURALES].fillna(0.0)
    datos = pd.concat([datos, construir_variables(datos)], axis=1)
    datos[OBJETIVO] = d[OBJETIVO].astype(int)
    return datos.reset_index(drop=True)


def particion(datos):
    """80 % entrenamiento, 20 % prueba, con la misma proporción de la clase en ambos."""
    return train_test_split(datos.index, test_size=0.2, stratify=datos[OBJETIVO], random_state=SEMILLA)

In [ ]:
def preprocesador(columnas, escalar):
    """Imputación + escala (si el modelo la necesita) para números; imputación + one-hot para categorías."""
    num = [c for c in columnas if c not in CATEGORICAS]
    cat = [c for c in columnas if c in CATEGORICAS]
    pasos = [("imputar", SimpleImputer(strategy="median"))]
    if escalar:
        pasos.append(("escalar", StandardScaler()))
    transformadores = [("num", Pipeline(pasos), num)]
    if cat:
        transformadores.append(("cat", Pipeline([
            ("imputar", SimpleImputer(strategy="constant", fill_value="Desconocido")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), cat))
    return ColumnTransformer(transformadores)


# nombre -> (familia, ¿necesita escala?)
FAMILIAS = {
    "Línea base (clase mayoritaria)": ("referencia", False),
    "Regresión logística": ("referencia", True),
    "SVM": ("SVM", True),
    "Bagging": ("Bagging", False),
    "Random Forest": ("Bagging", False),
    "AdaBoost": ("Boosting", False),
    "Gradient Boosting": ("Boosting", False),
    "Stacking": ("Stacking", True),
}
MODELOS = list(FAMILIAS)


def estimador(nombre, hp=None):
    """Estimador con sus hiperparámetros ajustados (hp: {modelo: {parámetro: valor}})."""
    hp = hp or {}
    p = dict(hp.get(nombre, {}))
    if nombre == "Línea base (clase mayoritaria)":
        return DummyClassifier(strategy="most_frequent")
    if nombre == "Regresión logística":
        return LogisticRegression(max_iter=5000, **p)
    if nombre == "SVM":
        return SVC(kernel="rbf", random_state=SEMILLA, **p)
    if nombre == "Bagging":
        hoja = p.pop("estimator__min_samples_leaf", 1)
        return BaggingClassifier(estimator=DecisionTreeClassifier(min_samples_leaf=hoja, random_state=SEMILLA),
                                 random_state=SEMILLA, n_jobs=-1, **p)
    if nombre == "Random Forest":
        return RandomForestClassifier(random_state=SEMILLA, n_jobs=-1, **p)
    if nombre == "AdaBoost":
        prof = p.pop("estimator__max_depth", 1)
        return AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=prof, random_state=SEMILLA),
                                  random_state=SEMILLA, **p)
    if nombre == "Gradient Boosting":
        return GradientBoostingClassifier(random_state=SEMILLA, **p)
    if nombre == "Stacking":
        base = [("svm", estimador("SVM", hp)), ("rf", estimador("Random Forest", hp)),
                ("gb", estimador("Gradient Boosting", hp))]
        return StackingClassifier(estimators=base, final_estimator=LogisticRegression(max_iter=5000),
                                  cv=StratifiedKFold(5, shuffle=True, random_state=SEMILLA), n_jobs=-1)
    raise KeyError(nombre)


def modelo(nombre, columnas, hp=None):
    """Pipeline completo: preprocesamiento + estimador."""
    return Pipeline([("pre", preprocesador(columnas, FAMILIAS[nombre][1])), ("modelo", estimador(nombre, hp))])


def puntaje(m, X):
    """Puntaje continuo para el ROC-AUC (probabilidad o, en SVM, la distancia al margen)."""
    if hasattr(m, "predict_proba"):
        try:
            return m.predict_proba(X)[:, 1]
        except AttributeError:
            pass
    return m.decision_function(X)


def importancia_permutacion_cv(nombre, columnas, X, y, hp, repeticiones=10):
    """Importancia por permutación medida en los pliegues de validación (nunca en la prueba)."""
    medidas = []
    for tr, va in CV5.split(X, y):
        m = modelo(nombre, columnas, hp).fit(X.iloc[tr][columnas], y.iloc[tr])
        r = permutation_importance(m, X.iloc[va][columnas], y.iloc[va], scoring="roc_auc",
                                   n_repeats=repeticiones, random_state=SEMILLA, n_jobs=-1)
        medidas.append(pd.Series(r.importances_mean, index=columnas))
    tabla = pd.concat(medidas, axis=1)
    return (pd.DataFrame({"importancia": tabla.mean(axis=1), "desvio_entre_pliegues": tabla.std(axis=1)})
            .sort_values("importancia", ascending=False))


def guardar_figura(fig, nombre):
    fig.savefig(IMG / nombre, dpi=150, bbox_inches="tight")


def leer_hp():
    return json.loads((MODELO_DIR / "hiperparametros.json").read_text(encoding="utf-8"))